In [1]:
from pathlib import Path

from functions import *

from minerva.transforms.transform import *
from minerva.transforms.random_transform import *

from minerva.data.readers import TiffReader, PNGReader
from minerva.data.datasets import SimpleDataset

from minerva.pipelines.lightning_pipeline import SimpleLightningPipeline
from lightning.pytorch.loggers.csv_logs import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning import Trainer
from torchmetrics import Accuracy, JaccardIndex, F1Score


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
repetition = 0
# pretrain_data = 'f3'
# finetune_data = 'f3'
pretrain_data = 'seam_ai'
finetune_data = 'seam_ai'
cap = 0.01
data_path = '/workspaces/shared_data/seam_ai_datasets/seam_ai_N'
# data_path = '/workspaces/shared_data/seismic/f3_segmentation'
batch_size = 2
num_epochs = 50
gpus = 0

In [3]:
torch.manual_seed(repetition)

In [4]:
if finetune_data == "f3" or finetune_data == "f3_N":
    padding = Padding(256, 704)
elif finetune_data == "seam_ai" or finetune_data == "seam_ai_N":
    padding = Padding(1008, 592)

In [5]:
image_path = f"{data_path}/images"
label_path = f"{data_path}/annotations"

train_data_reader = TiffReader(path=f"{image_path}/train")
train_label_reader = PNGReader(path=f"{label_path}/train")

val_data_reader = TiffReader(path=f"{image_path}/val")
val_label_reader = PNGReader(path=f"{label_path}/val")

test_data_reader = TiffReader(path=f"{image_path}/test")
test_label_reader = PNGReader(path=f"{label_path}/test") 

In [6]:
train_dataset = SimpleDataset(
    readers=[
        train_data_reader,
        train_label_reader,
    ],
    transforms=padding,
    return_single=False,
)

val_dataset = SimpleDataset(
    readers=[
        val_data_reader,
        val_label_reader,
    ],
    transforms=padding,
    return_single=False,
)

test_dataset = SimpleDataset(
    readers=[
        test_data_reader,
        test_label_reader,
    ],
    transforms=padding,
    return_single=False,
) 

In [7]:
data_module = CapDataModule(
    cap_train=cap,
    cap_val=1,
    cap_test=1,
    seed=repetition,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    test_dataset=test_dataset,
    batch_size=batch_size,
    drop_last=True,
    shuffle_train=True,
)

In [8]:
models_list = get_models_files()
len(models_list)

3

In [9]:
models_list = get_models_files()
model_example = models_list[0]
model_example

{'model_name': 'V0_pre_seam_ai_N_train_seam_ai_N_cap_100%',
 'repetition': '0',
 'pretrain_data': 'seam_ai_N',
 'train_data': 'seam_ai_N',
 'cap': '100%',
 'ckpt_file': 'ckpt/train/0/V0_pre_seam_ai_N_train_seam_ai_N_cap_100%/seam_ai_N/epoch=3-step=560.ckpt'}

In [10]:
model = get_eval_model(
    pretrain_data=model_example['pretrain_data'],
    import_path=model_example['ckpt_file'],
    learning_rate=0.001
    )

Model loaded from ckpt/train/0/V0_pre_seam_ai_N_train_seam_ai_N_cap_100%/seam_ai_N/epoch=3-step=560.ckpt


In [ ]:
# Métricas
num_classes = 6

metrics = {
        "mIoU": JaccardIndex(
            num_classes=6, average="macro", task="multiclass"
        ),
        "acc": Accuracy(num_classes=6, task="multiclass"),
        "f1-weighted": F1Score(
            num_classes=6, task="multiclass", average="weighted"
        ),
    }

In [12]:
ckpt_path = "./ckpt/test"
logs_path = "./logs/test"

log_dir = Path(logs_path) / model_example["model_name"] / finetune_data
ckpt_dir = Path(ckpt_path) / model_example["model_name"] / finetune_data

In [13]:
log_dir = Path(logs_path) 
ckpt_dir = Path(ckpt_path) 
logger = CSVLogger(log_dir, name=model_example["model_name"], version=finetune_data)
ckpt_callback = ModelCheckpoint(
    save_top_k=1, save_last=True, dirpath=ckpt_dir, mode="min", monitor="val_loss"
)

trainer = Trainer(
    accelerator="gpu",
    logger=logger,
    callbacks=[ckpt_callback],
    max_epochs=num_epochs,
    strategy="auto",
    devices=[gpus],
    check_val_every_n_epoch=2,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [14]:
metrics = {
        "mIoU": JaccardIndex(
            num_classes=6, average="macro", task="multiclass"
        ),
        "acc": Accuracy(num_classes=6, task="multiclass"),
        "f1-weighted": F1Score(
            num_classes=6, task="multiclass", average="weighted"
        ),
    }

pipeline = SimpleLightningPipeline(
    model=model,
    trainer=trainer,
    log_dir=log_dir,
    save_run_status=False,
    seed=repetition,
    apply_metrics_per_sample=False,
    classification_metrics=metrics,
    # save_predictions=True,
    # classification_reduce="squeeze",
)

In [ ]:
pipeline.run(data_module, task="evaluate")

** Seed set to: 0 **


/workspaces/Seismic-Byol/Minerva-Dev/minerva/pipelines/lightning_pipeline.py:278: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/vscode/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 100/100 [00:07<00:00, 13.50it/s]
Running classification metrics...
Metrics saved to /workspaces/Seismic-Byol/dev-seismic-byol/logs/test/metrics_2025-04-28-17-35-16814f1a61.yaml


{'classification': {'mIoU': [0.7197016477584839],
  'acc': [0.9208077192306519],
  'f1-weighted': [0.9201039671897888]}}